# DistilBERT fine-tuning for binary campaign classification

This notebook fine-tunes **`distilbert-base-uncased`** with a sequence-classification head on campaign **text** (name and blurb) for **successful** versus **failed** outcomes. Training and validation rows come from `data/features/train.csv` and `data/features/val.csv`.

Metrics logged during training include **accuracy**, **F1**, **precision**, **recall**, and **ROC AUC** on the validation split.


In [1]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np
import torch
import evaluate
import os
from sklearn.metrics import roc_auc_score

## Data preparation

CSV splits are filtered to successful versus failed campaigns, **`text`** is built from **name** and **blurb**, and **`label`** is binary. **`train_df`** / **`val_df`** (text and label only) back the Hugging Face `Dataset` objects.


In [2]:
def prepare_text_split(df):
    """Filter rows, build text and label for sequence classification."""
    df = df[df["state"].isin(["successful", "failed"])].copy()
    df["text"] = df["name"].fillna("") + " " + df["blurb"].fillna("")
    df["label"] = (df["state"] == "successful").astype(int)
    df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
    return df[["text", "label"]]


train_df = prepare_text_split(pd.read_csv("data/features/train.csv"))
val_df = prepare_text_split(pd.read_csv("data/features/val.csv"))

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

## Pretrained model and tokenizer

`AutoTokenizer` and **`AutoModelForSequenceClassification`** are loaded from **`distilbert-base-uncased`** with **two** labels. The classification head is trained together with the encoder unless a local checkpoint overwrites weights in a later cell.


In [3]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Tokenization

Each text field is tokenized with **truncation**, **padding to `max_length=128`**, and batched **`map`** on both splits.


In [4]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/66096 [00:00<?, ? examples/s]

Map:   0%|          | 0/22032 [00:00<?, ? examples/s]

## Tensor dataset format

The **text** column is removed after tokenization. **`set_format("torch")`** keeps **`input_ids`**, **`attention_mask`**, and **`label`** as tensors for the `Trainer`.


In [5]:
train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")

## Metrics logged each evaluation epoch

The **`compute_metrics`** callback converts logits to hard predictions and softmax **positive-class** scores for **ROC AUC**, alongside accuracy, F1, precision, and recall.


In [6]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs_pos = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()[:, 1]
    roc_auc = roc_auc_score(labels, probs_pos)

    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    precision = precision_metric.compute(predictions=predictions, references=labels)
    recall = recall_metric.compute(predictions=predictions, references=labels)

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "roc_auc": float(roc_auc),
    }

## Training arguments

New checkpoints are written under **`./distilbert_results`**. Optimization uses **two** epochs, per-device batch size **8**, learning rate **2e-5**, weight decay **0.01**, and epoch-level evaluation and saving. The run with the best validation **F1** is retained when **`load_best_model_at_end`** applies.


In [7]:
training_args = TrainingArguments(
    output_dir="./distilbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=1
)

## Checkpoint loading versus training

A finished **`DistilBertForSequenceClassification`** run can be restored from **`./distilbert_results/checkpoint-16524`** when that directory exists (standard Hugging Face `Trainer` layout: `config.json`, **`model.safetensors`**, tokenizer files). The following code cell replaces the in-memory **`model`** with **`AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_DIR)`** before the `Trainer` is constructed. Set **`RUN_TRAINING = True`** to run **`trainer.train()`** from the current weights (base initialization or checkpoint). Leave **`RUN_TRAINING = False`** to skip training and continue with evaluation and prediction on the validation split.


In [8]:
# Path to a Trainer checkpoint (matches the saved run folder name; adjust if cwd differs).
CHECKPOINT_DIR = "./distilbert_results/checkpoint-16524"
RUN_TRAINING = False

_loaded_from_checkpoint = False
if CHECKPOINT_DIR and os.path.isdir(CHECKPOINT_DIR):
    model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT_DIR)
    _loaded_from_checkpoint = True
    print("Loaded model weights from:", CHECKPOINT_DIR)
else:
    if not RUN_TRAINING:
        print(
            "CHECKPOINT_DIR is missing or not a directory; set RUN_TRAINING = True "
            "to fine-tune from the base checkpoint in the model cell, or fix CHECKPOINT_DIR."
        )


Loaded model weights from: ./distilbert_results/checkpoint-16524


## Trainer instantiation

The Hugging Face **`Trainer`** wraps the model, datasets, tokenizer, and **`compute_metrics`** for training and evaluation loops.


In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


/var/folders/0f/1mt731m54b51gldjgy_zszqr0000gn/T/ipykernel_20331/3012789700.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## Fine-tuning loop

When **`RUN_TRAINING`** is **True**, **`trainer.train()`** runs for the configured number of epochs. Otherwise training is skipped in favor of checkpoint or base weights, as described above.


In [10]:
if RUN_TRAINING:
    trainer.train()
else:
    # trainer.train()
    if _loaded_from_checkpoint:
        print("Skipped trainer.train(); using weights loaded from checkpoint.")
    else:
        print(
            "Skipped trainer.train(); model is still the base init from distilbert-base-uncased "
            "(no valid CHECKPOINT_DIR and RUN_TRAINING is False)."
        )


Skipped trainer.train(); using weights loaded from checkpoint.


## Evaluation on the validation split

**`trainer.evaluate()`** returns loss and **`eval_*`** metrics on the validation dataset.


In [11]:
results = trainer.evaluate()
results

{'eval_loss': 0.5350070595741272,
 'eval_model_preparation_time': 0.0006,
 'eval_accuracy': 0.7673384168482208,
 'eval_f1': 0.8125228586058079,
 'eval_precision': 0.7781981224604175,
 'eval_recall': 0.8500153045607591,
 'eval_roc_auc': 0.8390960749777329,
 'eval_runtime': 98.898,
 'eval_samples_per_second': 222.775,
 'eval_steps_per_second': 27.847}

## Per-example predictions and ROC AUC

**`trainer.predict`** emits validation logits; argmax yields class labels, and **ROC AUC** uses softmax probabilities for the successful class.


In [12]:
pred_output = trainer.predict(val_dataset)
logits_val_pred = pred_output.predictions
preds = np.argmax(logits_val_pred, axis=-1)
val_probs = torch.softmax(torch.from_numpy(logits_val_pred), dim=-1).numpy()[:, 1]
print(
    "ROC AUC (val predict):",
    float(roc_auc_score(pred_output.label_ids, val_probs)),
)
print(preds[:10])

ROC AUC (val predict): 0.8390960749777329
[1 0 1 1 1 1 1 1 1 0]
